# Esercizio
Classificazione multi-classe del dataset cifar10. (train-val-test)
Effettuare un confronto nelle performance fra rete neurale artificiale (ANN) e rete neurale convoluzionale (CNN).

Caricare poi un modello tramite Tensorflow con il comando `tf.keras.applications.[nomeModello]` per effettuare transfer learning su una nuova CNN, ed effettuare l'addestramento e la validazione del nuovo modello. Confrontare poi tutti e 3 i modelli con le metriche di classificazione.

In [1]:
# Import principali per dati e modelli
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
# Caricamento CIFAR-10
tf.random.set_seed(42)
np.random.seed(42)

(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# normalizzazione
X_train_full = X_train_full.astype("float32") / 255.
X_test = X_test.astype("float32") / 255.

# Nomi delle classi CIFAR-10
class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=42
)

## 2. Rete Neurale Artificiale (ANN Standard)

In [10]:
ann_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Flatten(),
    
    # Blocco 1
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    # Blocco 2
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    # Blocco 3
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    
    # Output Layer
    tf.keras.layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Addestramento esteso a 25 epoche
ann_model.fit(
    X_train, y_train, 
    epochs=25, 
    batch_size=64, 
    validation_data=(X_val, y_val)
)

Epoch 1/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.3164 - loss: 1.9559 - val_accuracy: 0.3306 - val_loss: 1.8391
Epoch 2/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.3829 - loss: 1.7169 - val_accuracy: 0.3760 - val_loss: 1.7082
Epoch 3/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4032 - loss: 1.6583 - val_accuracy: 0.3944 - val_loss: 1.6650
Epoch 4/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4156 - loss: 1.6336 - val_accuracy: 0.4012 - val_loss: 1.6725
Epoch 5/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4170 - loss: 1.6294 - val_accuracy: 0.3840 - val_loss: 1.6827
Epoch 6/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4326 - loss: 1.5908 - val_accuracy: 0.4336 - val_loss: 1.5764
Epoch 7/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4374 - loss: 1.5778 - val_accuracy: 0.4072 - val_loss: 1.6335
Epoch 8/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4366 - loss: 1.5736 - val_accuracy: 0

## 3. Rete Neurale Convoluzionale (CNN Custom)

In [12]:
cnn_improved = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    
    # Data Augmentation integrata nel modello
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    
    # Blocco 1
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    # Blocco 2
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.3),
    
    # Dense Classifier
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax')
])

cnn_improved.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Aumenta le epoche a 20-30 per dare tempo alla rete di convergere
cnn_improved.fit(X_train, y_train, epochs=25, batch_size=64, validation_data=(X_val, y_val))

Epoch 1/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 36s 48ms/step - accuracy: 0.4296 - loss: 1.6363 - val_accuracy: 0.5472 - val_loss: 1.2730
Epoch 2/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 35s 50ms/step - accuracy: 0.5546 - loss: 1.2496 - val_accuracy: 0.5892 - val_loss: 1.1669
Epoch 3/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 36s 51ms/step - accuracy: 0.6029 - loss: 1.1264 - val_accuracy: 0.5362 - val_loss: 1.3679
Epoch 4/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 36s 51ms/step - accuracy: 0.6347 - loss: 1.0402 - val_accuracy: 0.6356 - val_loss: 1.0400
Epoch 5/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 36s 51ms/step - accuracy: 0.6545 - loss: 0.9858 - val_accuracy: 0.6896 - val_loss: 0.9011
Epoch 6/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 36s 51ms/step - accuracy: 0.6734 - loss: 0.9360 - val_accuracy: 0.6892 - val_loss: 0.9174
Epoch 7/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 35s 50ms/step - accuracy: 0.6881 - loss: 0.8976 - val_accuracy: 0.6802 - val_loss: 0.9562
Epoch 8/25
704/704 ━━━━━━━━━━━━━━━━━━━━ 36s 52ms/step - accuracy: 0.6964 - loss: 0.8756 - 

## 4. Transfer Learning

In [17]:
# 4. Transfer Learning (con MobileNetV2)
inputs = tf.keras.layers.Input(shape=(32, 32, 3))

# Resizing a 96x96 e preprocessing nativo MobileNetV2
x = tf.keras.layers.Resizing(96, 96)(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

# Caricamento del modello base pre-addestrato
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Congelamento iniziale dei pesi

# Definizione del nuovo classificatore
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

transfer_model = tf.keras.Model(inputs, outputs)

# FASE 1: Addestramento rapido della testa (5 epoche)
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Fase 1: Addestramento testa del classificatore...")
transfer_model.fit(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

# FASE 2: Fine-Tuning (Sblocco degli ultimi layer con Learning Rate ridotto)
base_model.trainable = True
for layer in base_model.layers[:100]:  # Congela i primi 100 layer di basso livello
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("\nFase 2: Fine-Tuning dei layer superiori...")
transfer_model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

Fase 1: Addestramento testa del classificatore...
Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 148s 104ms/step - accuracy: 0.0988 - loss: 2.3138 - val_accuracy: 0.0952 - val_loss: 2.3029
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 105s 74ms/step - accuracy: 0.0989 - loss: 2.3030 - val_accuracy: 0.0952 - val_loss: 2.3028
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 103s 73ms/step - accuracy: 0.0981 - loss: 2.3029 - val_accuracy: 0.0952 - val_loss: 2.3028
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 102s 72ms/step - accuracy: 0.0981 - loss: 2.3029 - val_accuracy: 0.0952 - val_loss: 2.3028
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 101s 72ms/step - accuracy: 0.0979 - loss: 2.3029 - val_accuracy: 0.0952 - val_loss: 2.3028

Fase 2: Fine-Tuning dei layer superiori...
Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 161s 111ms/step - accuracy: 0.2580 - loss: 2.0371 - val_accuracy: 0.1052 - val_loss: 2.3032
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 154s 109ms/step - accuracy: 0.3962 - loss: 1.6926 - val_accuracy: 0.1

## 5. Confronto Finale sui Dati di Test

In [16]:
from sklearn.metrics import classification_report

models = {'ANN': ann_model, 'CNN Custom': cnn_model, 'Transfer Learning': transfer_model}

for name, model in models.items():
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"--- {name} ---")
    print(f"Test Accuracy: {test_acc:.4f}")
    
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    print(classification_report(y_test, y_pred, target_names=class_names))

--- ANN ---
Test Accuracy: 0.4635
              precision    recall  f1-score   support

    airplane       0.47      0.53      0.50      1000
  automobile       0.70      0.39      0.50      1000
        bird       0.33      0.43      0.37      1000
         cat       0.41      0.22      0.29      1000
        deer       0.38      0.52      0.44      1000
         dog       0.50      0.28      0.35      1000
        frog       0.53      0.46      0.49      1000
       horse       0.48      0.55      0.51      1000
        ship       0.50      0.68      0.58      1000
       truck       0.50      0.58      0.54      1000

    accuracy                           0.46     10000
   macro avg       0.48      0.46      0.46     10000
weighted avg       0.48      0.46      0.46     10000

--- CNN Custom ---
Test Accuracy: 0.6809
              precision    recall  f1-score   support

    airplane       0.71      0.71      0.71      1000
  automobile       0.80      0.81      0.80      1000
   